# SP4 — Coupled Twin Training (Kaggle / Colab GPU)

Drives the **real** SP4 training on a free GPU using the tested pipeline in
`scripts/sp4_pipeline.py`. Runs identically on CPU (slow) via `pick_device()`.

**Prerequisites**
1. VitalDB DUA accepted at https://vitaldb.net (needed for the drug-track pull).
2. This repo available in the runtime (git clone, or a Kaggle dataset copy).

**Recommended runtime:** Kaggle Notebook, GPU T4/P100 (more stable + 30 GPU-h/week
than free Colab). Colab works too. `Run All` after setting the repo path below.

## 1. Setup

In [ ]:
!pip -q install vitaldb torch pandas pyarrow

In [ ]:
import os, sys
# On Kaggle/Colab this clones your GitHub repo. If the repo already exists
# locally, set the SP4_REPO env var to its path to skip the clone.
REPO_URL = ('https://github.com/junaidraza343/'
            'AI-Digital-Twin-for-Perioperative-Risk-Prediction-Drug-Response-Simulation.git')
REPO = os.environ.get('SP4_REPO', '/kaggle/working/FYP')
if not os.path.isdir(REPO):
    !git clone {REPO_URL} {REPO}
sys.path.insert(0, REPO); os.chdir(REPO)
assert os.path.isdir(os.path.join(REPO, 'twin')), 'clone failed or wrong path'
print('repo ready at', REPO)

In [ ]:
import torch
from twin.models.deepnet import pick_device
device = pick_device()
print('device:', device, '| cuda:', torch.cuda.is_available())

## 2. Build the SP4 cohort (drug-track re-pull)

Pulls the drug-track + ART cases from VitalDB and writes `results/sp4_cohort.csv`.
First run downloads data (slow); reruns hit the on-disk cache. Verify the printed
coverage is non-trivial — if propofol coverage is ~0, the pump track name in
`twin/config.py` (`Orchestra/PPF20_RATE`) may need to be a TCI track for this subset.

In [ ]:
!python -m scripts.measure_drug_coverage

In [ ]:
import pandas as pd, twin.config as c
cohort_ids = pd.read_csv(c.RESULTS_DIR / 'sp4_cohort.csv')['caseid'].tolist()
eligible = pd.read_csv(c.RESULTS_DIR / 'eligible_cases.csv').set_index('caseid')
cohort = [(cid, eligible.loc[cid].to_dict())
          for cid in cohort_ids if cid in eligible.index]
print(f'SP4 cohort: {len(cohort)} cases')

## 3. Stage 1 — distillation (calibration head <- teacher deltas)

Fits a per-case teacher delta through the differentiable twin, then trains the
coupled model's calibration head to predict it from window features.

> **Cost note:** the teacher is fit per case with batch size 1 (`fit_deltas_batch`
> called once per case over its full length). This is the dominant wall-clock cost.
> For a large cohort, reduce `teacher_iters` or batch cases (see the deferred
> batching + norepi-hoist optimizations noted in the plan's Task 8).

In [ ]:
from scripts.sp4_pipeline import run_stage1_cohort
model, prep, cols, info = run_stage1_cohort(
    cohort, teacher_iters=200, epochs=300, lr=1e-2, latent_dim=32,
    device=device, save_path=c.RESULTS_DIR / 'coupled_stage1.pt')
print(info)

## 4. Stage 2 — end-to-end fine-tune (BCE + MAP reconstruction)

In [ ]:
from scripts.sp4_pipeline import run_stage2_cohort
model, s2 = run_stage2_cohort(
    cohort, model, prep, cols, epochs=100, lr=5e-3,
    save_path=c.RESULTS_DIR / 'coupled_stage2.pt')
print(s2)

## 5. Evaluation

Personalized (learned delta) vs population (zero delta) MAP reconstruction RMSE
per case, per-delta identifiability across the cohort, and a what-if monotonicity
sanity check. Writes `results/sp4_personalization.csv`.

In [ ]:
import numpy as np, torch
from scripts.sp4_pipeline import load_case
from twin.eval.personalization import (
    per_patient_map_rmse, delta_identifiability, whatif_monotonic)

rows, deltas_all = [], []
zero = torch.zeros(1, 6, dtype=torch.float64)
for cid, static in cohort:
    case = load_case(cid, static, device=device)
    if case is None:
        continue
    x = torch.tensor(prep.transform(case['windows'][cols]),
                     dtype=torch.float32, device=device)
    with torch.no_grad():
        _, delta, _ = model(x)
    delta_case = delta.mean(0, keepdim=True).double().cpu()
    p, n, o = case['prop'].cpu(), case['norepi'].cpu(), case['observed'].cpu()
    rmse_pers = per_patient_map_rmse([case['patient']], p, n, o, delta_case)[0]
    rmse_pop = per_patient_map_rmse([case['patient']], p, n, o, zero)[0]
    rows.append({'caseid': cid, 'rmse_personalized': float(rmse_pers),
                 'rmse_population': float(rmse_pop)})
    deltas_all.append(delta_case)

res = pd.DataFrame(rows)
res.to_csv(c.RESULTS_DIR / 'sp4_personalization.csv', index=False)
print('median RMSE  personalized=%.3f  population=%.3f' % (
    res.rmse_personalized.median(), res.rmse_population.median()))
print('win rate (personalized < population): %.1f%%' % (
    100 * (res.rmse_personalized < res.rmse_population).mean()))
print('delta identifiability (std per V1,V2,V3,ke0,EC50,gamma):')
print(delta_identifiability(torch.cat(deltas_all)))
print('what-if monotonic (first case):',
      whatif_monotonic(load_case(*cohort[0], device=device)['patient']))

## 6. Download artifacts

`results/coupled_stage1.pt`, `results/coupled_stage2.pt`, `results/sp4_personalization.csv`.
On Kaggle these persist in `/kaggle/working`; commit the notebook output or download them.

### SP2 hand-off
To upgrade to waveforms, pass a CNN-GRU encoder as `CoupledTwin(..., encoder=WaveformEncoder(...))`;
the calibration head, twin, losses, and this pipeline are unchanged.